In [1]:
import torch
import pickle
import numpy as np
import pandas as pd
import os
import json 
from os.path import dirname

#RS
from torch.utils.data import WeightedRandomSampler



root_path = dirname(os.getcwd()) + "/SEPH_OUTCOME"

pd.set_option("display.max_columns", None)
data_dir = root_path + "/data/datasets/original/"
data_dir_processed = root_path + "/data/datasets/processed/"
data_dir_graphs = root_path + "/data/datasets/graphs_repair/"

print("CWD:", os.getcwd())
print(root_path, data_dir, data_dir_processed, data_dir_graphs, sep="\n")

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# device = "cpu"

CWD: /home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_OUTCOME
/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_OUTCOME
/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_OUTCOME/data/datasets/original/
/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_OUTCOME/data/datasets/processed/
/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_OUTCOME/data/datasets/graphs_repair/


In [2]:
with open("data/dataset_features.json", 'r') as file:
    datasets_info = json.load(file)

In [3]:
list(datasets_info.keys())

['BPIC11_f1', 'sepsis_cases_1', 'sepsis_cases_4', 'BPIC15_common']

In [4]:
dataset = "BPIC11_f1" #decide which dataset to work on

In [5]:
#if dataset.startswith("BPIC15"):
#    with open("data/dataset_features.json", 'r') as file:
#        dataset_info = json.load(file)["BPIC15_common"]
#else:
#    with open("data/dataset_features.json", 'r') as file:
#        dataset_info = json.load(file)[dataset]


with open("data/dataset_features.json", 'r') as file:
        dataset_info = json.load(file)[dataset]

In [6]:
categorical_columns = dataset_info["categorical"]
real_value_columns = dataset_info["numerical"]

In [7]:
tab_all = pd.read_csv(data_dir_processed+dataset+"_processed_all.csv")
tab_all.head()

,Diagnosis,Treatment code,Diagnosis code,Specialism code,Diagnosis Treatment Combination ID,Age,CaseID,Label,Activity,Producer code,Section,Specialism code.1,group,Number of executions,time:timestamp,timesincemidnight,month,weekday,hour,timesincelastevent,timesincecasestart,event_nr,open_cases
0,maligniteit cervix,TC103,M13,SC61,DTC376907,33,0,regular,AC410100,SRTH,Section 5,SC61,Radiotherapy,1,1.104692e+09,1380,1,6,23,0.0,0.0,1,5
1,maligniteit cervix,TC103,M13,SC61,DTC376907,33,0,regular,AC419100,SRTH,Section 5,SC61,Radiotherapy,1,1.104692e+09,1380,1,6,23,0.0,0.0,2,5
2,maligniteit cervix,TC103,M13,SC61,DTC376907,33,0,regular,AC10107,SGEH,Section 2,SC7,Nursing ward,1,1.104865e+09,1380,1,1,23,0.0,2880.0,3,5
3,maligniteit cervix,TC103,M13,SC61,DTC376907,33,0,regular,339486E,SGEC,Section 2,SC7,Obstetrics & Gynaecology clinic,1,1.104865e+09,1380,1,1,23,0.0,2880.0,4,5
4,maligniteit cervix,TC103,M13,SC61,DTC376907,33,0,regular,AC410100,SGEH,Section 2,SC7,Nursing ward,1,1.104865e+09,1380,1,1,23,0.0,2880.0,5,5


In [8]:
import random

torch.manual_seed(0)
torch.cuda.manual_seed(0)
random.seed(0)
np.random.seed(0)

In [9]:
tab_test = pd.read_csv(f"data/datasets/processed/{dataset}_processed_test.csv")
minority_lengths = tab_test[tab_test["Label"]=="deviant"].groupby("CaseID").size()
total_minority = len(minority_lengths)
q90 = int(np.ceil(minority_lengths.quantile(0.9)))
MaxPrefix = min(40,q90)
print(f"90th‐percentile threshold: {q90:.2f}  MaxPrefix: {MaxPrefix}")

90th‐percentile threshold: 30.00  MaxPrefix: 30


In [10]:
with open(data_dir_graphs + dataset + "_TRAIN_repair.pkl", "rb") as f:
    X_train = pickle.load(f)
with open(data_dir_graphs + dataset + "_VALID_repair.pkl", "rb") as f:
    X_valid = pickle.load(f)
with open(data_dir_graphs + dataset + "_TEST4_repair.pkl", "rb") as f:
    X_test = pickle.load(f)

X_tests = {}
for L in range(1, MaxPrefix + 1):
    fname = f"{dataset}_TEST{L}_repair.pkl"
    path  = os.path.join(data_dir_graphs, fname)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Expected file not found: {path}")
    with open(path, "rb") as f:
        X_tests[L] = pickle.load(f)
    print(f"Loaded {len(X_tests[L])} graphs for prefix length {L} from {fname}")

Loaded 225 graphs for prefix length 1 from BPIC11_f1_TEST1_repair.pkl
Loaded 216 graphs for prefix length 2 from BPIC11_f1_TEST2_repair.pkl
Loaded 211 graphs for prefix length 3 from BPIC11_f1_TEST3_repair.pkl
Loaded 205 graphs for prefix length 4 from BPIC11_f1_TEST4_repair.pkl
Loaded 199 graphs for prefix length 5 from BPIC11_f1_TEST5_repair.pkl
Loaded 198 graphs for prefix length 6 from BPIC11_f1_TEST6_repair.pkl
Loaded 196 graphs for prefix length 7 from BPIC11_f1_TEST7_repair.pkl
Loaded 195 graphs for prefix length 8 from BPIC11_f1_TEST8_repair.pkl
Loaded 195 graphs for prefix length 9 from BPIC11_f1_TEST9_repair.pkl
Loaded 193 graphs for prefix length 10 from BPIC11_f1_TEST10_repair.pkl
Loaded 193 graphs for prefix length 11 from BPIC11_f1_TEST11_repair.pkl
Loaded 192 graphs for prefix length 12 from BPIC11_f1_TEST12_repair.pkl
Loaded 191 graphs for prefix length 13 from BPIC11_f1_TEST13_repair.pkl
Loaded 191 graphs for prefix length 14 from BPIC11_f1_TEST14_repair.pkl
Loaded 191

In [11]:

from torch_geometric.data import Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.transforms import ToUndirected, NormalizeFeatures

transform = ToUndirected()

with torch.no_grad():
        for i in range(len(X_train)):
                X_train[i] = transform(X_train[i])
        for i in range(len(X_valid)):
                X_valid[i] = transform(X_valid[i])
        for L, graphs_L in X_tests.items():
                for i in range(len(graphs_L)):
                        graphs_L[i] = transform(graphs_L[i])
    


In [12]:
edge_types = set()
node_types = set()
for i in range(len(X_train)):
    n, edge_type = X_train[i].metadata()
    for x in n:
        node_types.add(x)
    for x in edge_type:
        edge_types.add(x)
for i in range(len(X_valid)):
    n, edge_type = X_valid[i].metadata()
    for x in n:
        node_types.add(x)
    for x in edge_type:
        edge_types.add(x)
for graphs_L in X_tests.values():
    for i in range(len(graphs_L)):
        n, edge_type = graphs_L[i].metadata()
        for x in n:
            node_types.add(x)
        for x in edge_type:
            edge_types.add(x)



In [13]:
node_types = list(node_types)
edge_types = list(edge_types)

In [14]:
node_types

['Diagnosis Treatment Combination ID',
 'Number of executions',
 'event_nr',
 'open_cases',
 'group',
 'Section',
 'Age',
 'weekday',
 'Activity',
 'Producer code',
 'timesincemidnight',
 'hour',
 'timesincelastevent',
 'month',
 'Diagnosis code',
 'Specialism code',
 'Treatment code',
 'Diagnosis',
 'Specialism code.1',
 'timesincecasestart']

In [15]:
edge_types

[('Specialism code', 'related_to', 'Specialism code'),
 ('Activity', 'related_to', 'Diagnosis'),
 ('Section', 'rev_related_to', 'Activity'),
 ('Specialism code.1', 'related_to', 'Specialism code.1'),
 ('Diagnosis Treatment Combination ID',
  'related_to',
  'Diagnosis Treatment Combination ID'),
 ('Activity', 'followed_by', 'Activity'),
 ('Diagnosis', 'rev_related_to', 'Activity'),
 ('Activity', 'related_to', 'Number of executions'),
 ('Specialism code', 'rev_related_to', 'Activity'),
 ('Diagnosis Treatment Combination ID', 'rev_related_to', 'Activity'),
 ('group', 'rev_related_to', 'Activity'),
 ('Treatment code', 'related_to', 'Treatment code'),
 ('Activity', 'related_to', 'Treatment code'),
 ('Activity', 'related_to', 'timesincemidnight'),
 ('Diagnosis code', 'rev_related_to', 'Activity'),
 ('event_nr', 'rev_related_to', 'Activity'),
 ('timesincelastevent', 'related_to', 'timesincelastevent'),
 ('Activity', 'related_to', 'open_cases'),
 ('Activity', 'related_to', 'timesincelastevent

## Hyperopt

In [16]:
print(f"PyTorch: {torch.__version__}")
#print(f"TorchVision: {torchvision.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

PyTorch: 2.6.0+cu124
CUDA Available: False


In [17]:
from ax.service.managed_loop import optimize

In [18]:
from torch_geometric.nn import (
    HeteroConv,
    global_mean_pool,
    GATv2Conv,
    SAGEConv,
    TransformerConv
)
from torch.nn import (
    ModuleList,
    Module,
    Linear
  )
from typing_extensions import Self

In [19]:
from torch_geometric.nn import HeteroConv, global_mean_pool, SAGEConv
from torch.nn import Module, ModuleList, Sequential, Linear, Dropout, BatchNorm1d, ReLU
import torch.nn.functional as F

class HGNN(Module):
    def __init__(self, nodes_relations, parameters):
        super().__init__()
        hid           = parameters["hid"]
        layers        = parameters["layers"]
        aggregation   = parameters["aggregation"]
        dropout_p     = parameters.get("dropout", 0.1)

        # 1) stack of hetero‐message‐passing layers
        self.convs = ModuleList()
        self.bns   = ModuleList()
        self.dps   = ModuleList()
        for _ in range(layers):
            # hetero‐conv over each relation
            conv = HeteroConv(
                { rel: SAGEConv((-1, -1), aggr=aggregation, out_channels=hid, normalize=False)
                  for rel in nodes_relations },
                aggr=aggregation,
            )
            self.convs.append(conv)
            # batchnorm + dropout for the hidden dim
            self.bns.append(BatchNorm1d(hid))
            self.dps.append(Dropout(dropout_p))

        # 2) final graph‐classification head: MLP hid→hid→1
        self.classifier = Sequential(
            Linear(hid, hid),
            ReLU(),
            BatchNorm1d(hid),
            Dropout(dropout_p),
            Linear(hid, 1),
        )

    def forward(self, batch):
        x_dict    = batch.x_dict
        edge_dict = batch.edge_index_dict

        # --- message‑passing with BN/ReLU/Dropout after each conv ---
        for conv, bn, dp in zip(self.convs, self.bns, self.dps):
            x_dict = conv(x_dict, edge_dict)

            # normalize + activate + drop only on the “Activity” embeddings
            act = x_dict["Activity"]
            act = bn(act)
            act = F.relu(act)
            act = dp(act)
            x_dict["Activity"] = act

            # for all other node types, just ReLU
            for nt, x in x_dict.items():
                if nt != "Activity":
                    x_dict[nt] = F.relu(x)

        # --- graph‑level readout on “Activity” nodes ---
        h_act  = x_dict["Activity"]
        pooled = global_mean_pool(h_act, batch["Activity"].batch)

        # --- final MLP head → logits ---
        logits = self.classifier(pooled).view(-1)
        return logits



    

In [20]:
from torcheval.metrics.functional import multiclass_accuracy, multiclass_f1_score
import torch.nn as nn
import time

In [21]:
#Weighted Random Sampling
#Pull out all labels into a single 1D tensor of 0/1
y_train = torch.cat([g.y for g in X_train]).long()
#count examples per class
class_counts = torch.bincount(y_train)
#Inverse frequency
class_weights = 1.0 / class_counts.float()

print("class_counts:", class_counts.tolist())
print("class_weights:", class_weights.tolist())


# number of negatives & positives
n_neg, n_pos = class_counts.tolist()

# the weight for positive class = n_neg / n_pos
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float, device=device)
print("Using BCEWithLogitsLoss pos_weight =", pos_weight.item())

#Assign each sample the weight of it's class
sample_weights = class_weights[y_train]
#create a sampler that draws 'len(sample_weights)' samples per epoch
sampler = WeightedRandomSampler(
     weights=sample_weights,
     num_samples=len(sample_weights),
     replacement=True,
 )

class_counts: [324, 234]
class_weights: [0.003086419776082039, 0.004273504484444857]
Using BCEWithLogitsLoss pos_weight = 1.384615421295166


In [22]:
from collections import Counter

# Draw 10,000 “indices” from the sampler
sampled_indices = list(WeightedRandomSampler(
    weights=sample_weights,
    num_samples=500,
    replacement=True
))

# Map each index back to its label
sampled_labels = [ y_train[idx].item() for idx in sampled_indices ]
print(Counter(sampled_labels))

Counter({0: 251, 1: 249})


In [23]:
from copy import deepcopy
from tqdm.notebook import tqdm

def train_hgnn(config, epochs=20):

    
    print(config)

    net = HGNN(
        parameters=config,
        nodes_relations=edge_types,
    )
    net = net.to(device)

    # loss for graph binary classification
    #loss_fn = nn.BCEWithLogitsLoss()
    loss_fn = nn.BCEWithLogitsLoss()

    #train_loader = DataLoader(X_train, batch_size=config["batch_size"], shuffle=True)
    train_loader = DataLoader(
        X_train,
        batch_size=config["batch_size"],
        sampler=sampler,     # ← use the balanced sampler
        shuffle=False,       # ← don’t shuffle when using sampler
    )


    valid_loader = DataLoader(X_valid, batch_size=config["batch_size"], shuffle=False)


    optimizer = torch.optim.Adam(net.parameters(), lr=config["lr"])

    best_model = None
    best_loss = float("inf")
    patience = 5
    pat_count = 0

    torch.cuda.empty_cache()

    for epoch in tqdm(range(0, epochs)):
        start_time = time.time()

        #print(f"Epoch: {epoch}\n")

        net.train()
        for _, x in enumerate(train_loader):
            x = x.to(device)

            optimizer.zero_grad()       

            logits = net(x) #shape [batch_size]
            labels = x.y.float() #shape [batch_size]
            loss = loss_fn(logits, labels)

            loss.backward()
            optimizer.step()

        #--validation--
        #running_loss = 0.0
        #correct = 0
        #total = 0
        running_loss = 0.0
        all_logits = []
        all_labels = []

        net.eval()
        with torch.no_grad():
            for x in valid_loader:
                x = x.to(device)
                logits = net(x)
                labels = x.y 

                running_loss +=loss_fn(logits, labels.float()).item()

                #compute binary predictions
                #preds = (torch.sigmoid(logits) > 0.5).long()
                #correct += (preds == labels).sum().item()
                #total += labels.size(0)

                #accumulate for AUC
                all_logits.append(logits.cpu())
                all_labels.append(labels.cpu())


        val_loss = running_loss / len(valid_loader)
        #val_acc = correct / total

        #Concatenate and compute AUC
        all_logits = torch.cat(all_logits)
        all_labels = torch.cat(all_labels).numpy()
        all_probs  = torch.sigmoid(all_logits).numpy()
        from sklearn.metrics import roc_auc_score
        val_auc = roc_auc_score(all_labels, all_probs)


        # Early stopping 
        if val_loss < best_loss:
            best_loss  = val_loss
            best_model = deepcopy(net)
            pat_count  = 0
        else:
            pat_count += 1
            if pat_count >= patience:
                break

    return best_model




In [24]:
def test_hgnn_multi(net):
    """
    Evaluate a trained HGNN over multiple prefix-length test sets.
    Uses global X_tests and the batch_size from net.parameters.
    Returns a dict mapping each prefix L -> AUC@L, plus the weighted-average.
    """
    net.eval()
    aucs   = []
    counts = []

    for L, graphs_L in X_tests.items():
        print(f"\n--- Testing prefix length L = {L} ---")
        loader = DataLoader(graphs_L, batch_size=128, shuffle=False)

        all_logits = []
        all_labels = []
        total_loss = 0.0
        loss_fn    = nn.BCEWithLogitsLoss()

        with torch.no_grad():
            for batch in loader:
                batch = batch.to(device)
                logits = net(batch)
                labels = batch.y

                total_loss += loss_fn(logits, labels.float()).item()
                all_logits.append(logits.cpu())
                all_labels.append(labels.cpu())

        # Aggregate
        avg_loss = total_loss / len(loader)
        all_logits = torch.cat(all_logits)
        all_labels = torch.cat(all_labels).numpy()
        all_probs  = torch.sigmoid(all_logits).numpy()
        
        from sklearn.metrics import roc_auc_score
        auc_L = roc_auc_score(all_labels, all_probs)
        n_L   = len(graphs_L)
        print(f"AUC@{L} = {auc_L:.4f}  (n_graphs={n_L}, avg_loss={avg_loss:.4f})")

        aucs.append(auc_L)
        counts.append(n_L)

    # Weighted-average AUC
    aucs   = np.array(aucs)
    counts = np.array(counts)
    weighted_auc = np.average(aucs, weights=counts)
    print(f"\n>>> Weighted-average AUC over prefixes 1–{len(aucs)}: {weighted_auc:.4f}")

    return {
        **{f"AUC@{L}": a for L, a in zip(X_tests.keys(), aucs)},
        "Weighted_AUC": weighted_auc
    }


In [25]:
from torch_geometric.loader import DataLoader
import torch.nn as nn
import torch

def test_hgnn(net):
    """
    Evaluate a trained HGNN (graph‑level classifier) on X_test.
    Returns a dict with test loss and accuracy.
    """
    test_loader = DataLoader(X_test, batch_size=128, shuffle=False)
    loss_fn = nn.BCEWithLogitsLoss()

    net.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    all_logits = []
    all_labels = []

    with torch.no_grad():
        for x in test_loader:
            x = x.to(device)
            logits = net(x)           
            labels = x.y              

            # accumulate loss
            total_loss += loss_fn(logits, labels.float()).item()

            # binary predictions & accuracy
            #preds = (torch.sigmoid(logits) > 0.5).long()
            #correct += (preds == labels).sum().item()
            #total += labels.size(0)

            # accumulate for AUC
            all_logits.append(logits.cpu())
            all_labels.append(labels.cpu())


    avg_loss = total_loss / len(test_loader)
    #accuracy = correct / total
    # compute AUC
    all_logits = torch.cat(all_logits)
    all_labels = torch.cat(all_labels).numpy()
    all_probs  = torch.sigmoid(all_logits).numpy()
    from sklearn.metrics import roc_auc_score
    test_auc = roc_auc_score(all_labels, all_probs)


    #print(f"Test loss: {avg_loss:.4f}, Test accuracy: {accuracy:.4f}")
    #return {"test_loss": avg_loss, "test_acc": accuracy}
    print(f"Test loss: {avg_loss:.4f}, Test ROC‑AUC: {test_auc:.4f}")
    return {"test_loss": avg_loss, "test_auc": test_auc}


In [26]:
# Calculate unique counts for categorical columns
list_unique = {col: len(tab_all[col].unique()) for col in categorical_columns}

#outputcat = {k : len(list_unique[k]) for k in list_unique}
outputcat = list_unique
outputreal = real_value_columns
print(outputcat)
print(outputreal)

{'Diagnosis': 105, 'Treatment code': 43, 'Diagnosis code': 11, 'Specialism code': 3, 'Diagnosis Treatment Combination ID': 799, 'CaseID': 1140, 'Activity': 193, 'Producer code': 52, 'Section': 7, 'Specialism code.1': 14, 'group': 24}
['Age', 'Number of executions', 'timesincemidnight', 'month', 'weekday', 'hour', 'timesincelastevent', 'timesincecasestart', 'event_nr', 'open_cases']


In [27]:
def train_evaluate(config):
    trained_net = train_hgnn(config, epochs=50)
    #return test_hgnn    #keep for faster execution when refining parameters
    return test_hgnn_multi(trained_net)

In [28]:
import logging

logging.getLogger("root").setLevel(logging.ERROR)

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [29]:
y_train = torch.cat([batch.y for batch in X_train]).float()
num_true = y_train.sum().item()
num_false = len(y_train) - num_true

# Assign weights to BCEWithLogitsLoss
pos_weight = num_false / num_true
pos_weight = torch.tensor([num_false / num_true], device=device)

print("pos_weight: ", pos_weight)

pos_weight:  tensor([1.3846])


In [30]:
sample_config = {
    "hid":          128, #128
    "layers":       2, #2
    "lr":           1e-3,
    "batch_size":   256, #256
    "aggregation": "mean",
}


# 1-epoch train just to exercise the code-path and see prints
net = train_hgnn(sample_config, epochs=1)

# run the test (with your debug prints enabled)
res = test_hgnn(net)
print("test_hgnn returned:", res)
print("\n")

# run the test, multi version
res = test_hgnn_multi(net)
print("test_hgnn returned:", res)


{'hid': 128, 'layers': 2, 'lr': 0.001, 'batch_size': 256, 'aggregation': 'mean'}


  0%|          | 0/1 [00:00<?, ?it/s]

Test loss: 0.6920, Test ROC‑AUC: 0.5000
test_hgnn returned: {'test_loss': 0.6919933259487152, 'test_auc': 0.5}



--- Testing prefix length L = 1 ---
AUC@1 = 0.4921  (n_graphs=225, avg_loss=0.6964)

--- Testing prefix length L = 2 ---
AUC@2 = 0.5046  (n_graphs=216, avg_loss=0.6941)

--- Testing prefix length L = 3 ---
AUC@3 = 0.5241  (n_graphs=211, avg_loss=0.6916)

--- Testing prefix length L = 4 ---
AUC@4 = 0.5229  (n_graphs=205, avg_loss=0.6910)

--- Testing prefix length L = 5 ---
AUC@5 = 0.4884  (n_graphs=199, avg_loss=0.6904)

--- Testing prefix length L = 6 ---
AUC@6 = 0.4923  (n_graphs=198, avg_loss=0.6906)

--- Testing prefix length L = 7 ---
AUC@7 = 0.4971  (n_graphs=196, avg_loss=0.6902)

--- Testing prefix length L = 8 ---
AUC@8 = 0.4971  (n_graphs=195, avg_loss=0.6901)

--- Testing prefix length L = 9 ---
AUC@9 = 0.4998  (n_graphs=195, avg_loss=0.6899)

--- Testing prefix length L = 10 ---
AUC@10 = 0.4889  (n_graphs=193, avg_loss=0.6898)

--- Testing prefix length L = 11 -

In [31]:
best_parameters, values, experiment, model = optimize(
    parameters=[
        {"name": "hid", "type": "choice", "values": [64,128,256,512], "value_type": "int", "is_ordered" : True,"sort_values":False},
        #{"name": "hid", "type": "choice", "values": [512], "value_type": "int", "is_ordered" : True,"sort_values":False},
        {"name": "layers", "type": "choice", "values": [2, 3, 4, 5], "value_type": "int", "is_ordered" : True, "sort_values":False},
        #{"name": "layers", "type": "choice", "values": [2], "value_type": "int", "is_ordered" : True, "sort_values":False},
        {"name": "lr", "type": "range", "bounds": [1e-4, 1e-1], "value_type": "float", "log_scale": True},
        {"name": "batch_size", "type": "choice", "values": [128,256,512], "value_type": "int", "is_ordered" : True,"sort_values":False}, 
        
        #{"name": "heads", "type": "choice", "values": [1,2], "value_type": "int", "is_ordered" : True,"sort_values":False},
        #{"name": "heads", "type": "choice", "values": [1], "value_type": "int", "is_ordered" : True,"sort_values":False},
        
        {"name": "aggregation", "type" : "choice", "values" :["sum", "mean", "max"], "value_type" : "str"}
        #{"name": "aggregation", "type" : "choice", "values" :["max"], "value_type" : "str"},
     
    ],
  
    evaluation_function=train_evaluate,
    objective_name='Weighted_AUC', #test_auc for single/multi switch
    arms_per_trial=1,
    minimize = False,
    random_seed = 123,
    total_trials = 30
)

print(best_parameters)
means, covariances = values
print(means)
print(experiment)

/home/matteo/Documents/GNN-test2/SEPH_MODELS/env3/lib/python3.10/site-packages/ax/service/utils/instantiation.py:248: AxParameterWarning: `is_ordered` is not specified for `ChoiceParameter` "aggregation". Defaulting to `False`  since the parameter is a string with more than 2 choices.. To override this behavior (or avoid this warning), specify `is_ordered` during `ChoiceParameter` construction. Note that choice parameters with exactly 2 choices are always considered ordered and that the user-supplied `is_ordered` has no effect in this particular case.
  return ChoiceParameter(
/home/matteo/Documents/GNN-test2/SEPH_MODELS/env3/lib/python3.10/site-packages/ax/service/utils/instantiation.py:248: AxParameterWarning: `sort_values` is not specified for `ChoiceParameter` "aggregation". Defaulting to `False` for parameters of `ParameterType` STRING. To override this behavior (or avoid this warning), specify `sort_values` during `ChoiceParameter` construction.
  return ChoiceParameter(
[INFO 06

{'hid': 64, 'layers': 4, 'lr': 0.001671979996274695, 'batch_size': 512, 'aggregation': 'sum'}


  0%|          | 0/50 [00:00<?, ?it/s]


--- Testing prefix length L = 1 ---
AUC@1 = 0.5387  (n_graphs=225, avg_loss=0.7777)

--- Testing prefix length L = 2 ---
AUC@2 = 0.5607  (n_graphs=216, avg_loss=0.7578)

--- Testing prefix length L = 3 ---
AUC@3 = 0.5672  (n_graphs=211, avg_loss=0.7373)

--- Testing prefix length L = 4 ---
AUC@4 = 0.5552  (n_graphs=205, avg_loss=0.7304)

--- Testing prefix length L = 5 ---
AUC@5 = 0.5256  (n_graphs=199, avg_loss=0.7262)

--- Testing prefix length L = 6 ---
AUC@6 = 0.5295  (n_graphs=198, avg_loss=0.7269)

--- Testing prefix length L = 7 ---
AUC@7 = 0.5362  (n_graphs=196, avg_loss=0.7229)

--- Testing prefix length L = 8 ---
AUC@8 = 0.5380  (n_graphs=195, avg_loss=0.7221)

--- Testing prefix length L = 9 ---
AUC@9 = 0.5404  (n_graphs=195, avg_loss=0.7218)

--- Testing prefix length L = 10 ---
AUC@10 = 0.5297  (n_graphs=193, avg_loss=0.7204)

--- Testing prefix length L = 11 ---
AUC@11 = 0.5300  (n_graphs=193, avg_loss=0.7203)

--- Testing prefix length L = 12 ---
AUC@12 = 0.5240  (n_gra

[INFO 06-27 17:01:54] ax.core.experiment: Attached data has some metrics ({'AUC@2', 'AUC@18', 'AUC@25', 'AUC@7', 'AUC@14', 'AUC@16', 'AUC@26', 'AUC@11', 'AUC@8', 'AUC@6', 'AUC@5', 'AUC@19', 'AUC@27', 'AUC@21', 'AUC@23', 'AUC@3', 'AUC@24', 'AUC@29', 'AUC@20', 'AUC@15', 'AUC@22', 'AUC@10', 'AUC@12', 'AUC@9', 'AUC@28', 'AUC@13', 'AUC@4', 'AUC@1', 'AUC@17', 'AUC@30'}) that are not among the metrics on this experiment. Note that attaching data will not automatically add those metrics to the experiment. For these metrics to be automatically fetched by `experiment.fetch_data`, add them via `experiment.add_tracking_metric` or update the experiment's optimization config.
[INFO 06-27 17:01:54] ax.service.managed_loop: Running optimization trial 2...
[ERROR 06-27 17:01:54] ax.core.observation: Data contains metric AUC@1 that has not been added to the experiment. You can either update the `optimization_config` or attach it as a tracking metric using `Experiment.add_tracking_metrics` or `AxClient.a

...


--- Testing prefix length L = 1 ---
AUC@1 = 0.8153  (n_graphs=225, avg_loss=0.6667)

--- Testing prefix length L = 2 ---
AUC@2 = 0.8568  (n_graphs=216, avg_loss=0.5881)

--- Testing prefix length L = 3 ---
AUC@3 = 0.8666  (n_graphs=211, avg_loss=0.6041)

--- Testing prefix length L = 4 ---
AUC@4 = 0.8681  (n_graphs=205, avg_loss=0.6293)

--- Testing prefix length L = 5 ---
AUC@5 = 0.8742  (n_graphs=199, avg_loss=0.6279)

--- Testing prefix length L = 6 ---
AUC@6 = 0.8791  (n_graphs=198, avg_loss=0.6286)

--- Testing prefix length L = 7 ---
AUC@7 = 0.8861  (n_graphs=196, avg_loss=0.6122)

--- Testing prefix length L = 8 ---
AUC@8 = 0.8930  (n_graphs=195, avg_loss=0.6091)

--- Testing prefix length L = 9 ---
AUC@9 = 0.8935  (n_graphs=195, avg_loss=0.6073)

--- Testing prefix length L = 10 ---
AUC@10 = 0.8957  (n_graphs=193, avg_loss=0.5957)

--- Testing prefix length L = 11 ---
AUC@11 = 0.8974  (n_graphs=193, avg_loss=0.5941)

--- Testing prefix length L = 12 ---
AUC@12 = 0.8976  (n_gra

[INFO 06-27 19:06:28] ax.core.experiment: Attached data has some metrics ({'AUC@2', 'AUC@18', 'AUC@25', 'AUC@7', 'AUC@14', 'AUC@16', 'AUC@26', 'AUC@11', 'AUC@8', 'AUC@6', 'AUC@5', 'AUC@19', 'AUC@27', 'AUC@21', 'AUC@23', 'AUC@3', 'AUC@24', 'AUC@29', 'AUC@20', 'AUC@15', 'AUC@22', 'AUC@10', 'AUC@12', 'AUC@9', 'AUC@28', 'AUC@13', 'AUC@4', 'AUC@1', 'AUC@17', 'AUC@30'}) that are not among the metrics on this experiment. Note that attaching data will not automatically add those metrics to the experiment. For these metrics to be automatically fetched by `experiment.fetch_data`, add them via `experiment.add_tracking_metric` or update the experiment's optimization config.


AUC@30 = 0.7738  (n_graphs=90, avg_loss=0.9662)

>>> Weighted-average AUC over prefixes 1–30: 0.8803


[ERROR 06-27 19:06:28] ax.core.observation: Data contains metric AUC@1 that has not been added to the experiment. You can either update the `optimization_config` or attach it as a tracking metric using `Experiment.add_tracking_metrics` or `AxClient.add_tracking_metrics`. Ignoring all data for metric AUC@1.
NoneType: None
[ERROR 06-27 19:06:28] ax.core.observation: Data contains metric AUC@10 that has not been added to the experiment. You can either update the `optimization_config` or attach it as a tracking metric using `Experiment.add_tracking_metrics` or `AxClient.add_tracking_metrics`. Ignoring all data for metric AUC@10.
NoneType: None
[ERROR 06-27 19:06:28] ax.core.observation: Data contains metric AUC@11 that has not been added to the experiment. You can either update the `optimization_config` or attach it as a tracking metric using `Experiment.add_tracking_metrics` or `AxClient.add_tracking_metrics`. Ignoring all data for metric AUC@11.
NoneType: None
[ERROR 06-27 19:06:28] ax.c

{'hid': 512, 'layers': 5, 'lr': 0.0001, 'batch_size': 128, 'aggregation': 'sum'}
{'Weighted_AUC': 0.8367014674412303}
Experiment(None)


In [32]:
from ax.service.utils.report_utils import exp_to_df

results = exp_to_df(experiment)
#results.sort_values(by="test_auc")
#results = results.sort_values(by="test_auc")
results.sort_values(by="Weighted_AUC")
results = results.sort_values(by="Weighted_AUC")
results.to_csv(f"results/{dataset}.csv", sep=",")

[WARNING 06-27 19:06:35] ax.service.utils.report_utils: Column reason missing for all trials. Not appending column.
